# FlashEats — Class 7 Challenge
## Model the Business Workflow with Data

### Client question
> **“Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI.”**

Do not repeat source discovery, retrieval, or data-cleaning work. Today the goal is to build a useful business-workflow model.

> **Solved version.** Every original prompt is kept; each TODO is filled in and a written answer follows
> each challenge. It runs from `Challenge_Solutions/` inside the project, which has the same `database/`,
> `data/` and `api/` folders as the classroom pack.

In [1]:
# Setup - needs pandas
import json, sqlite3
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

# find the folder that holds database/flasheats.db (the project root, one level up from this notebook)
BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "database" / "flasheats.db").exists())
print("Using pack:", BASE.name)

Using pack: sst-FDE-ass-2-flasheats-pipeline


In [2]:
con=sqlite3.connect(BASE/"database"/"flasheats.db")
orders=pd.read_sql("SELECT * FROM orders",con)
customers=pd.read_sql("SELECT * FROM customers",con)
restaurants=pd.read_sql("SELECT * FROM restaurants",con)
drivers=pd.read_sql("SELECT * FROM drivers",con)
tickets=pd.read_csv(BASE/"data"/"support_tickets.csv")
customer_actions=pd.read_csv(BASE/"data"/"customer_app_actions.csv")
interventions=pd.read_csv(BASE/"data"/"order_interventions.csv")
outcomes=pd.read_csv(BASE/"data"/"order_outcomes.csv")
with open(BASE/"data"/"class7_model_brief.json") as f: model_brief=json.load(f)
print("orders",orders.shape,"actions",customer_actions.shape,"interventions",interventions.shape,"outcomes",outcomes.shape)
print("Project KPI:",model_brief["project_kpi"])

orders (1603, 13) actions (2365, 6) interventions (430, 6) outcomes (1600, 6)
Project KPI: Reduce late delivery rate


# Challenge 1 — Reconstruct the order lifecycle

Choose 3 orders:
- one delivered on time,
- one delivered late,
- one with an intervention.

Build a timeline with:

`event_time | event_type | actor | source_system`

Include as many lifecycle events as the data supports.

### Hint
Start from one `order_id`, collect events from each source, then sort by time.

In [3]:
base_orders = orders.drop_duplicates("order_id", keep="first")
late_order = outcomes[outcomes.late_flag == 1].order_id.iloc[0]
on_time_order = outcomes[outcomes.late_flag == 0].order_id.iloc[0]
intervention_order = interventions.order_id.iloc[0]
print("late", late_order, "on-time", on_time_order, "intervention", intervention_order)

def build_order_timeline(order_id):
    o = base_orders[base_orders.order_id == order_id].iloc[0]
    rows = [
        (o.created_at, "ORDER_CREATED", f"customer {o.customer_id}", "orders (SQL)"),
        (o.promised_eta, "PROMISED_ETA (the promise, not an event)", "platform", "orders (SQL)"),
        (o.pickup_at, "PICKED_UP", f"driver {o.driver_id}", "orders (SQL)"),
        (o.actual_delivery_at, "DELIVERED", f"driver {o.driver_id}", "orders (SQL)"),
    ]
    for a in customer_actions[customer_actions.order_id == order_id].itertuples():
        rows.append((a.action_at, a.action_type, "customer", "customer_app_actions.csv"))
    for t in tickets[tickets.order_id == order_id].drop_duplicates().itertuples():
        rows.append((t.created_at, f"SUPPORT_TICKET ({t.category})", "customer", "support_tickets.csv"))
    for i in interventions[interventions.order_id == order_id].itertuples():
        rows.append((i.intervention_at, f"INTERVENTION: {i.intervention_type} ({i.reason})", i.initiated_by, "order_interventions.csv"))
    timeline = pd.DataFrame(rows, columns=["event_time", "event_type", "actor", "source_system"])
    timeline["event_time"] = pd.to_datetime(timeline.event_time, format="mixed", errors="coerce")
    return timeline.dropna(subset=["event_time"]).sort_values("event_time").reset_index(drop=True)

for label, oid in [("LATE", late_order), ("ON TIME", on_time_order), ("WITH INTERVENTION", intervention_order)]:
    out = outcomes[outcomes.order_id == oid].iloc[0]
    print(f"\n== {label}: {oid} -> {out.outcome_bucket}, delay {out.delay_min} min")
    display(build_order_timeline(oid))

late O00001 on-time O00003 intervention O00781

== LATE: O00001 -> delivered_late, delay 10.82 min


,event_time,event_type,actor,source_system
0,2026-08-13 12:56:00.000000,ORDER_CREATED,customer C0168,orders (SQL)
1,2026-08-13 13:08:00.000000,INTERVENTION: DRIVER_REASSIGNMENT (driver_unavailable),dispatch,order_interventions.csv
2,2026-08-13 13:21:00.000000,ETA_VIEWED,customer,customer_app_actions.csv
3,2026-08-13 13:35:49.825561,PICKED_UP,driver D103,orders (SQL)
4,2026-08-13 14:11:36.000000,"PROMISED_ETA (the promise, not an event)",platform,orders (SQL)
5,2026-08-13 14:20:36.000000,SUPPORT_OPENED,customer,customer_app_actions.csv
6,2026-08-13 14:22:25.035899,DELIVERED,driver D103,orders (SQL)



== ON TIME: O00003 -> delivered_on_time, delay -4.41 min


,event_time,event_type,actor,source_system
0,2026-08-01 19:35:00.000000,ORDER_CREATED,customer C0855,orders (SQL)
1,2026-08-01 19:57:50.820366,PICKED_UP,driver D039,orders (SQL)
2,2026-08-01 20:13:28.061191,DELIVERED,driver D039,orders (SQL)
3,2026-08-01 20:17:52.941967,"PROMISED_ETA (the promise, not an event)",platform,orders (SQL)



== WITH INTERVENTION: O00781 -> delivered_late, delay 13.07 min


,event_time,event_type,actor,source_system
0,2026-08-22 12:33:00.000000,ORDER_CREATED,customer C0103,orders (SQL)
1,2026-08-22 12:54:00.000000,ETA_VIEWED,customer,customer_app_actions.csv
2,2026-08-22 12:57:00.000000,INTERVENTION: PRIORITY_DISPATCH (late_risk),operations,order_interventions.csv
3,2026-08-22 13:11:17.309762,PICKED_UP,driver D103,orders (SQL)
4,2026-08-22 13:48:36.000000,"PROMISED_ETA (the promise, not an event)",platform,orders (SQL)
5,2026-08-22 14:01:40.289785,DELIVERED,driver D103,orders (SQL)


**Answer - Challenge 1**

`build_order_timeline()` pulls one order's events from four sources (orders table, app actions, support tickets,
interventions) and sorts them by time. The timelines show what the orders table alone can't: in the intervention
example (O00781), ops gave the order priority dispatch for "late risk" *after it was placed but before pickup* - and
it was still 13 minutes late.

Lifecycle events the data **does not** support: when the restaurant accepted the order, when the food was ready,
and when the driver arrived at the restaurant - so the gap between "created" and "picked up" is a black box.

# Challenge 2 — Define the canonical project model

Your model must support:
1. customer → orders
2. order → customer interactions
3. order → support interactions
4. order → interventions
5. order → outcome

For each table, document:
- primary key,
- important foreign keys,
- grain.

Then explain why this model is better for the project than mirroring every source-system table.

In [4]:
sources={"orders":orders,"customer_actions":customer_actions,"support_tickets":tickets,"interventions":interventions,"outcomes":outcomes}
for name,df in sources.items():
    print(name,df.shape); display(df.head(2))

# check each table's grain: is the key really unique?
grain = pd.DataFrame([
    ("orders", "order_id", len(orders), orders.order_id.nunique()),
    ("customers", "customer_id", len(customers), customers.customer_id.nunique()),
    ("customer_actions", "action_id", len(customer_actions), customer_actions.action_id.nunique()),
    ("support_tickets", "ticket_id", len(tickets), tickets.ticket_id.nunique()),
    ("interventions", "intervention_id", len(interventions), interventions.intervention_id.nunique()),
    ("outcomes", "order_id", len(outcomes), outcomes.order_id.nunique()),
], columns=["table", "key", "rows", "unique_keys"])
grain["key_is_unique"] = grain.rows == grain.unique_keys

# foreign keys: does every child row point at a real order / customer?
fk = pd.DataFrame([
    ("orders.customer_id -> customers", base_orders.customer_id.isin(customers.customer_id).mean()),
    ("customer_actions.order_id -> orders", customer_actions.order_id.isin(base_orders.order_id).mean()),
    ("support_tickets.order_id -> orders", tickets.order_id.isin(base_orders.order_id).mean()),
    ("interventions.order_id -> orders", interventions.order_id.isin(base_orders.order_id).mean()),
    ("outcomes.order_id -> orders", outcomes.order_id.isin(base_orders.order_id).mean()),
], columns=["relationship", "coverage"])
display(grain); display(fk.round(4))

orders (1603, 13)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear


customer_actions (2365, 6)


,action_id,order_id,customer_id,action_type,action_at,channel
0,ACT-00001,O01044,C0249,ETA_VIEWED,2026-08-05T22:11:00,mobile_app
1,ACT-00002,O01334,C0692,ETA_VIEWED,2026-08-01T17:48:00,mobile_app


support_tickets (202, 5)


...

,intervention_id,order_id,intervention_type,intervention_at,initiated_by,reason
0,INT-00001,O00781,PRIORITY_DISPATCH,2026-08-22T12:57:00,operations,late_risk
1,INT-00002,O01476,CUSTOMER_CREDIT,2026-08-01T23:46:36.200640,support,support_resolution


outcomes (1600, 6)


,order_id,final_status_norm,delivered_flag,late_flag,delay_min,outcome_bucket
0,O00001,delivered,1,1.0,10.82,delivered_late
1,O00002,delivered,1,1.0,3.97,delivered_late


,table,key,rows,unique_keys,key_is_unique
0,orders,order_id,1603,1600,False
1,customers,customer_id,900,900,True
2,customer_actions,action_id,2365,2365,True
3,support_tickets,ticket_id,202,201,False
4,interventions,intervention_id,430,430,True
5,outcomes,order_id,1600,1600,True


,relationship,coverage
0,orders.customer_id -> customers,1.0000
1,customer_actions.order_id -> orders,1.0000
2,support_tickets.order_id -> orders,0.9851
3,interventions.order_id -> orders,1.0000
4,outcomes.order_id -> orders,1.0000


**Answer - Challenge 2: the canonical project model**

| Table | Grain | Primary key | Foreign keys | Supports |
|---|---|---|---|---|
| `customer` | one customer | `customer_id` | - | customer -> orders |
| `order` | one order (duplicates removed) | `order_id` | `customer_id`, `restaurant_id`, `driver_id` | the spine of the model |
| `customer_interaction` | one app action or support ticket | `action_id` / `ticket_id` | `order_id`, `customer_id` | order -> customer and support interactions |
| `intervention` | one intervention | `intervention_id` | `order_id` | order -> interventions |
| `order_outcome` | one order | `order_id` | `order_id` (1:1 with order) | order -> outcome (late / on time, delay) |

Key checks: `orders` and `support_tickets` are **not** unique on their keys as delivered (3 duplicate orders, 1
duplicate ticket), so the model removes those first; every action, intervention and outcome points at a real order.
3 tickets have no order_id at all.

**Why this is better than mirroring every source table:** the sources are organised by *system* (order platform,
app analytics, support desk, ops tools). The project question is about the *order journey*, and the KPI is measured
per order. Putting the order in the middle, with interactions, interventions and the outcome hanging off it, means
every question in this notebook is one join away - instead of stitching five system-shaped tables together each time.

# Challenge 3 — Build interaction → intervention → outcome

Create one order-level table containing:

`order_id, customer_id, support_opened, cancel_attempted, intervention_count, intervention_types, final_status, late_flag, delay_min`

Answer:
1. How many late orders had support interaction?
2. How many orders received intervention?
3. Which intervention is most common?
4. Which frustrated journeys had no intervention?

### Hint
Aggregate one-to-many tables before joining them to order-level outcomes.

In [5]:
actions_by_order=(customer_actions.groupby("order_id").agg(action_count=("action_id","count")).reset_index())

# aggregate every one-to-many table to ONE row per order before joining - so the join can't multiply orders
action_flags = customer_actions.pivot_table(index="order_id", columns="action_type", values="action_id",
                                            aggfunc="count", fill_value=0).reset_index()
ticket_orders = set(tickets.drop_duplicates().order_id.dropna())
iv = (interventions.groupby("order_id")
      .agg(intervention_count=("intervention_id", "count"),
           intervention_types=("intervention_type", lambda s: ", ".join(sorted(set(s)))))
      .reset_index())

model = (base_orders[["order_id", "customer_id"]]
         .merge(outcomes[["order_id", "final_status_norm", "late_flag", "delay_min"]], on="order_id", how="left")
         .merge(action_flags, on="order_id", how="left")
         .merge(iv, on="order_id", how="left"))
model["support_opened"] = (model["SUPPORT_OPENED"].fillna(0) > 0) | model["order_id"].isin(ticket_orders)
model["cancel_attempted"] = model["CANCEL_ATTEMPTED"].fillna(0) > 0
model["intervention_count"] = model["intervention_count"].fillna(0).astype(int)
model["intervention_types"] = model["intervention_types"].fillna("")
model = model.rename(columns={"final_status_norm": "final_status"})[
    ["order_id", "customer_id", "support_opened", "cancel_attempted", "intervention_count",
     "intervention_types", "final_status", "late_flag", "delay_min"]]

print(len(model), "rows,", model.order_id.nunique(), "unique orders  (the joins didn't multiply anything)")
display(model.head())

late = model[model.late_flag == 1]
print(f"\n1. Late orders with a support interaction: {late.support_opened.sum()} of {len(late)} ({late.support_opened.mean():.1%})")
print(f"2. Orders that received an intervention: {(model.intervention_count > 0).sum()}")
print("3. Most common intervention:"); display(interventions.intervention_type.value_counts())
frustrated = model[(model.support_opened | model.cancel_attempted) & (model.intervention_count == 0)]
print(f"4. Frustrated journeys (support opened or cancel attempted) with NO intervention: {len(frustrated)}, "
      f"of which late: {(frustrated.late_flag == 1).sum()}")
display(frustrated.head(10))

1600 rows, 1600 unique orders  (the joins didn't multiply anything)


,order_id,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types,final_status,late_flag,delay_min
0,O00001,C0168,True,False,1,DRIVER_REASSIGNMENT,delivered,1.0,10.82
1,O00002,C0043,False,False,0,,delivered,1.0,3.97
2,O00003,C0855,False,False,0,,delivered,0.0,-4.41
3,O00004,C0229,False,False,0,,cancelled,NaN,NaN
4,O00005,C0040,False,False,0,,delivered,1.0,21.02



1. Late orders with a support interaction: 258 of 843 (30.6%)


2. Orders that received an intervention: 430
3. Most common intervention:


intervention_type
DRIVER_REASSIGNMENT    155
RESTAURANT_CONTACT     116
PRIORITY_DISPATCH       95
CUSTOMER_CREDIT         64
Name: count, dtype: int64

4. Frustrated journeys (support opened or cancel attempted) with NO intervention: 219, of which late: 192


,order_id,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types,final_status,late_flag,delay_min
36,O00037,C0346,True,False,0,,delivered,1.0,15.45
39,O00040,C0374,True,False,0,,delivered,1.0,25.55
40,O00041,C0008,True,False,0,,delivered,1.0,5.40
41,O00042,C0862,True,False,0,,delivered,1.0,5.67
46,O00047,C0285,True,False,0,,delivered,1.0,28.11
48,O00049,C0544,True,False,0,,delivered,1.0,1.75
70,O00071,C0391,True,False,0,,delivered,1.0,11.74
72,O00073,C0750,True,False,0,,delivered,1.0,13.55
76,O00077,C0157,True,False,0,,delivered,1.0,26.54
79,O00080,C0229,True,False,0,,delivered,1.0,22.51


**Answer - Challenge 3**

1. **258 of 843 late orders (30.6%)** had a support interaction (support opened in the app or a ticket).
2. **430 orders** received at least one intervention.
3. The most common intervention is **DRIVER_REASSIGNMENT** (155), then restaurant contact (116), priority dispatch
   (95) and customer credit (64).
4. **219 frustrated journeys** - the customer opened support or tried to cancel - got **no intervention at all**, and
   192 of those were late. These are the clearest missed chances for ops.

# Challenge 4 — Select 3–5 business metrics

Project KPI: **Reduce Late Delivery Rate**.

For each chosen metric, document:
- metric name,
- formula,
- grain,
- why it matters,
- relationship to the project KPI.

At least one metric must represent:
- an outcome,
- a customer interaction,
- an intervention.

In [6]:
valid_o = model[model.late_flag.notna()]
late = valid_o[valid_o.late_flag == 1]
metrics = pd.DataFrame([
    ("Late Delivery Rate", "Outcome (the KPI)", "late delivered / delivered with a delivery time", "order",
     f"{valid_o.late_flag.mean():.1%}"),
    ("Median lateness of late orders", "Outcome (severity)", "median(delay_min) where late", "order",
     f"{late.delay_min.median():.1f} min"),
    ("Late-order support contact rate", "Customer interaction", "late orders with support opened or a ticket / late orders", "order",
     f"{late.support_opened.mean():.1%}"),
    ("Late-order intervention rate", "Intervention", "late orders with at least one intervention / late orders", "order",
     f"{(late.intervention_count > 0).mean():.1%}"),
    ("Cancel-attempt rate on late orders", "Customer interaction", "late orders with a cancel attempt / late orders", "order",
     f"{late.cancel_attempted.mean():.1%}"),
], columns=["metric", "type", "formula", "grain", "value"])
metrics

,metric,type,formula,grain,value
0,Late Delivery Rate,Outcome (the KPI),late delivered / delivered with a delivery time,order,56.4%
1,Median lateness of late orders,Outcome (severity),median(delay_min) where late,order,8.0 min
2,Late-order support contact rate,Customer interaction,late orders with support opened or a ticket / late orders,order,30.6%
3,Late-order intervention rate,Intervention,late orders with at least one intervention / late orders,order,27.0%
4,Cancel-attempt rate on late orders,Customer interaction,late orders with a cancel attempt / late orders,order,1.2%


**Answer - Challenge 4: five metrics**

| Metric | Why it matters | Relationship to the KPI |
|---|---|---|
| **Late Delivery Rate** (outcome) | the promise the business is judged on | *is* the KPI |
| **Median lateness of late orders** (outcome) | 8 minutes late and 60 minutes late are not the same failure | shows whether a fall in the KPI is real or just orders scraping in |
| **Late-order support contact rate** (customer interaction) | the cost of lateness in support load | falls when the KPI falls, *if* lateness is what drives contacts |
| **Late-order intervention rate** (intervention) | how often ops acts on late orders | the lever ops controls; should rise before the KPI falls |
| **Cancel-attempt rate on late orders** (customer interaction) | the strongest sign of a lost customer | small today (1.2%), but a warning signal if it grows |

# Challenge 5 — Investigate the workflow with joins and aggregations

Answer at least three:

A. Do orders with support interactions have higher delay?  
B. What is late rate with vs without intervention?  
C. Which intervention type is associated with the lowest late rate?  
D. Which restaurants contribute the largest number of late orders?  
E. Which journeys show support interaction + intervention + still late?

For every answer, add one sentence:

> **What does this tell the business, and what does it NOT prove?**

In [7]:
v = model[model.late_flag.notna()].copy()
v["late"] = v.late_flag == 1
v["has_intervention"] = v.intervention_count > 0

print("A. Delay with vs without a support interaction")
display(v.groupby("support_opened").agg(orders=("order_id", "size"), median_delay=("delay_min", "median"), late_rate=("late", "mean")).round(3))

print("B. Late rate with vs without an intervention")
display(v.groupby("has_intervention").agg(orders=("order_id", "size"), late_rate=("late", "mean"), median_delay=("delay_min", "median")).round(3))

print("C. Late rate by intervention type (one row per intervention)")
iv_outcome = interventions.merge(v[["order_id", "late", "delay_min"]], on="order_id")
display(iv_outcome.groupby("intervention_type").agg(interventions=("intervention_id", "size"), late_rate=("late", "mean"),
                                                    median_delay=("delay_min", "median")).sort_values("late_rate").round(3))

print("D. Restaurants with the most late orders")
rest = v.merge(base_orders[["order_id", "restaurant_id"]], on="order_id")
display(rest.groupby("restaurant_id").agg(late_orders=("late", "sum"), orders=("order_id", "size"), late_rate=("late", "mean"))
        .sort_values("late_orders", ascending=False).head(5).round(3))

print("E. Journeys with support interaction + intervention + still late")
e = v[v.support_opened & v.has_intervention & v.late]
print(len(e), "orders")
display(e.head(5))

A. Delay with vs without a support interaction


,orders,median_delay,late_rate
support_opened,,,
False,1203,-0.250,0.486
True,292,12.285,0.884


B. Late rate with vs without an intervention


,orders,late_rate,median_delay
has_intervention,,,
False,1091,0.564,1.39
True,404,0.564,1.54


C. Late rate by intervention type (one row per intervention)


,interventions,late_rate,median_delay
intervention_type,,,
PRIORITY_DISPATCH,86,0.512,0.33
RESTAURANT_CONTACT,108,0.528,0.80
DRIVER_REASSIGNMENT,147,0.578,1.60
CUSTOMER_CREDIT,63,0.667,2.93


D. Restaurants with the most late orders


,late_orders,orders,late_rate
restaurant_id,,,
R050,22,35,0.629
R004,20,30,0.667
R024,20,25,0.800
R023,20,32,0.625
R030,20,38,0.526


E. Journeys with support interaction + intervention + still late
67 orders


,order_id,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types,final_status,late_flag,delay_min,late,has_intervention
0,O00001,C0168,True,False,1,DRIVER_REASSIGNMENT,delivered,1.0,10.82,True,True
7,O00008,C0223,True,False,1,CUSTOMER_CREDIT,delivered,1.0,26.86,True,True
32,O00033,C0536,True,False,1,RESTAURANT_CONTACT,delivered,1.0,14.32,True,True
57,O00058,C0604,True,False,1,CUSTOMER_CREDIT,delivered,1.0,2.70,True,True
78,O00079,C0387,True,False,1,PRIORITY_DISPATCH,delivered,1.0,1.54,True,True


**Answer - Challenge 5** (association, not causation)

- **A.** Orders with a support interaction have a median delay of **+12.3 min**; orders without one, **-0.25 min**.
  *Tells:* support contact is a strong signal that an order is running late. *Doesn't prove:* that contacting
  support makes anything worse (or better) - customers contact support *because* the order is late.
- **B.** Late rate is **56.4% with an intervention and 56.4% without**. *Tells:* today's interventions show no
  measurable effect on lateness. *Doesn't prove:* they don't work - ops targets orders that are already in trouble,
  so the two groups aren't comparable. Only a controlled test would show it.
- **C.** Priority dispatch (51.2%) and restaurant contact (52.8%) have the lowest late rates; customer credit the
  highest (66.7%). *Tells:* which interventions are *associated* with better outcomes. *Doesn't prove:* that one is
  better - credits are mostly given *after* a late delivery, as compensation, so their high late rate is expected.
- **D.** R050 (22 late orders), then R004, R024, R023 and R030 (20 each) contribute the most late orders. *Tells:* where to start
  looking. *Doesn't prove:* the restaurant is at fault - busier restaurants simply have more orders, and we can't
  tell restaurant delay from driver delay.
- **E.** **67 orders** had a support interaction *and* an intervention and were *still* late. *Tells:* the response
  often comes too late to save the order. *Doesn't prove:* the intervention was wrong - it may have prevented a worse delay.

# Challenge 6 — Connect the model to the KPI

Create a one-page project view:

`PROJECT KPI → OUTCOME METRIC → WORKFLOW/DRIVER METRICS → INTERVENTIONS → DATA SOURCES/EVENTS`

Answer:
1. Which metrics are directly controllable by operations?
2. Which are outcomes?
3. Which missing event limits the model most?
4. What would you instrument next?

Final deliverable:
- core entities,
- key events,
- relationships,
- 3–5 metrics,
- KPI linkage,
- one modelling limitation.

**Answer - Challenge 6: one-page project view**

```
PROJECT KPI        Reduce Late Delivery Rate
      |
OUTCOME METRIC     Late Delivery Rate (56.4%)  +  median lateness of late orders
      |
WORKFLOW METRICS   support contact rate on late orders (30.6%), cancel attempts,
      |            time from order to pickup and from pickup to delivery
INTERVENTIONS      reassignment, priority dispatch, restaurant contact, credit
      |            (late-order intervention rate: 27%)
DATA SOURCES       orders (SQL), app actions, support tickets, interventions, outcomes
```

1. **Directly controllable by operations:** the interventions (how many, which type, how early) and the
   late-order intervention rate.
2. **Outcomes:** late delivery rate and median lateness.
3. **The missing event that limits the model most:** **driver arrived at the restaurant** (and food ready). Without
   it, everything between "order created" and "picked up" is one black box, so we can't tell restaurant delay from
   driver delay.
4. **What to instrument next:** driver arrival at the restaurant, food-ready time from the restaurant, and the ETA
   each time dispatch changes it.

**Final deliverable**

- **Core entities:** customer, order, restaurant, driver.
- **Key events:** order created, picked up, delivered; customer actions; support tickets; interventions.
- **Relationships:** customer 1-to-many orders; order 1-to-many interactions; order 1-to-many interventions; order 1-to-1 outcome.
- **3-5 metrics:** the five in Challenge 4.
- **KPI linkage:** the view above.
- **One modelling limitation:** `order_outcomes.csv` marks 37 delivered orders as "unknown" because the orders table
  has no delivery time - the driver app has those times (Class 5), so the model could recover them.

# Final reflection

A strong solution does not produce the largest schema.

It produces the **smallest useful model that explains the workflow and supports the project KPI**.